# Split datasets into train and test

To ensure the train and test datasets are different enough to prevent leakage, AAI will be calculated for all pairs of sequences in the combined geNomad/proGenomes dataset.

In [39]:
! pip install polars pandas duckdb --quiet

## All-vs-all protein searches

In [1]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data")
PROCESS_DIR = ROOT_DIR.joinpath("processing")
GENOMAD_PROGENOMES_COMBINED = PROCESS_DIR.joinpath("genomad_progenomes_combined")

### Combine proGenomes and geNomad proteins into one file for `mmseqs`

In [2]:
GENOMAD_FAA = PROCESS_DIR.joinpath("checkamg_annotate_outputs_1.1/checkamg_annotate_genomad_dataset/wdir/filtered_input/filtered_faa_by_cds/single_contig_proteins.faa")
PROGENOMES_FAA = PROCESS_DIR.joinpath("checkamg_annotate_outputs_1.1/checkamg_annotate_progenomes_dataset/wdir/filtered_input/filtered_faa_by_cds/single_contig_proteins.faa")
COMBINED_FAA = GENOMAD_PROGENOMES_COMBINED.joinpath("genomad_progenomes_combined.faa")

In [3]:
! cat {GENOMAD_FAA} {PROGENOMES_FAA} > {COMBINED_FAA}

In [4]:
! grep -c ">"  {GENOMAD_FAA} {PROGENOMES_FAA} {COMBINED_FAA}

/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_genomad_dataset/wdir/filtered_input/filtered_faa_by_cds/single_contig_proteins.faa:11723480
/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/checkamg_annotate_outputs_1.1/checkamg_annotate_progenomes_dataset/wdir/filtered_input/filtered_faa_by_cds/single_contig_proteins.faa:5102331
/storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/genomad_progenomes_combined.faa:16825811


### Create an `mmseqs` sequence database

In [6]:
MMSEQS_DIR = GENOMAD_PROGENOMES_COMBINED.joinpath("mmseqs")

In [7]:
! mkdir -p {MMSEQS_DIR}

In [8]:
! mmseqs createdb {COMBINED_FAA} {MMSEQS_DIR.joinpath("genomad_progenomes_combined.faa.mmseqs")}

createdb /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/genomad_progenomes_combined.faa /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.mmseqs 

MMseqs Version:                    	18.8cc5c
Database type                      	0
Shuffle input database             	true
Createdb mode                      	0
Write lookup file                  	1
Offset of numeric ids              	0
Threads                            	256
Compressed                         	0
Mask residues                      	0
Mask residues probability          	0.9
Mask lower case residues           	0
Mask lower letter repeating N times	0
Use GPU                            	0
Verbosity                          	3

Converting sequences
[16825793] 21s 39mss
Time for merging to genomad_progenomes_combined.faa.mmseqs_h: 0h 0m 3s 528ms
Time for merging to genomad_progenomes_comb

### Run `mmseqs search`
This takes 1-2 days with 100 CPUs if using max sensitivity (`7.5`). Instead, `--start-sens 4 --sens-steps 3 -s 7.5` is used for a slightly less sensitive search than `-s 7.5`, but still faster than just using the maximum sensitivity and more sensitive than the fast `-s 4.0` or default `-s 5.7`. This way will take 1-3 hours using 100 CPUs.

    nohup mmseqs search \
        /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.mmseqs \
        /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.mmseqs \
        /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.m8 \
        /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.tmp \
        --start-sens 4 --sens-steps 3 -s 7.5 \
        --min-seq-id 0.0 \
        --threads 100 \
        > ./logs/train_test_split/mmseqs_search_genomad_progenomes_combined.log &

In [3]:
ALIGNMENTS_TSV = GENOMAD_PROGENOMES_COMBINED.joinpath("genomad_progenomes_combined.alignments.tsv")

In [10]:
! mmseqs convertalis \
    {MMSEQS_DIR.joinpath("genomad_progenomes_combined.faa.mmseqs")} \
    {MMSEQS_DIR.joinpath("genomad_progenomes_combined.faa.mmseqs")} \
    {MMSEQS_DIR.joinpath("genomad_progenomes_combined.m8")} \
    {ALIGNMENTS_TSV} \
    --format-mode 4 \
    --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qlen,tlen,qcov,tcov \
    --threads 32

convertalis /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.mmseqs /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.faa.mmseqs /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/mmseqs/genomad_progenomes_combined.m8 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/genomad_progenomes_combined.alignments.tsv --format-mode 4 --format-output query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qlen,tlen,qcov,tcov --threads 32 

MMseqs Version:        	18.8cc5c
Substitution matrix    	aa:blosum62.out,nucl:nucleotide.out
Alignment format       	4
Format alignment output	query,target,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue,bits,qlen,tlen,qcov,tcov
Translation table    

### Load alignments, write to parquet, and delete TSV to save space

In [4]:
TMP_DIR = GENOMAD_PROGENOMES_COMBINED.joinpath("tmp_duckdb")
TMP_DIR.mkdir(exist_ok=True)

In [5]:
ALIGNMENTS_PARQUET = GENOMAD_PROGENOMES_COMBINED.joinpath("genomad_progenomes_combined.alignments.parquet")

In [13]:
import duckdb

con = duckdb.connect()

con.execute(f"""
PRAGMA threads=64;
PRAGMA memory_limit='800GB';
PRAGMA temp_directory='{TMP_DIR}';

COPY (
  SELECT *
  FROM read_csv_auto(
    '{ALIGNMENTS_TSV}',
    delim='\t',
    header=true
  )
) TO '{ALIGNMENTS_PARQUET}'
(FORMAT PARQUET, CODEC 'ZSTD');
""")
con.close()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Check to make sure the file was copied correctly

In [14]:
import duckdb

if not ALIGNMENTS_TSV.exists():
    raise FileNotFoundError(f"TSV not found: {ALIGNMENTS_TSV}")
if not ALIGNMENTS_PARQUET.exists():
    raise FileNotFoundError(f"Parquet not found: {ALIGNMENTS_PARQUET}")

con = duckdb.connect() # in-memory is fine for counts
con.execute("PRAGMA threads=64;")
con.execute("PRAGMA memory_limit='800GB';")
con.execute("PRAGMA temp_directory='/storage2/scratch/kosmopoulos/tmp_duckdb';")

n_tsv = con.execute(
    """
    SELECT count(*)
    FROM read_csv_auto(?, delim='\t', header=true)
    """,
    [str(ALIGNMENTS_TSV)],
).fetchone()[0]

n_parquet = con.execute(
    "SELECT count(*) FROM read_parquet(?)",
    [str(ALIGNMENTS_PARQUET)],
).fetchone()[0]

print(f"n_tsv:    {n_tsv:,}")
print(f"n_parquet:{n_parquet:,}")

if n_tsv == n_parquet:
    print("Counts match. Deleting TSV.")
    ALIGNMENTS_TSV.unlink()
    print("Deleted.")
else:
    raise RuntimeError("Counts do not match. TSV NOT deleted.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

n_tsv:    2,528,623,315
n_parquet:2,528,623,315
Counts match. Deleting TSV.
Deleted.


## Calculate AAI from mmseqs results

Using the script `compute_aai_self.py` located in `accessory_scripts`. This took about 15 minutes using 50 threads.

In [6]:
SCRIPTS_DIR = Path("./accessory_scripts")

In [7]:
AAI_OUTDIR = GENOMAD_PROGENOMES_COMBINED.joinpath("aai")

In [17]:
! python3 \
    {SCRIPTS_DIR.joinpath("compute_aai_self.py")} \
    --ptn_path {COMBINED_FAA} \
    --alignments_file {ALIGNMENTS_PARQUET} \
    --output_dir {AAI_OUTDIR} \
    --threads 50

2026-07-10 12:27:52 | Memory limit set to 1578 GB
2026-07-10 12:28:02 | Scanned 5,000,000 proteins
2026-07-10 12:28:12 | Scanned 10,000,000 proteins
2026-07-10 12:28:22 | Scanned 15,000,000 proteins
2026-07-10 12:28:26 | Genome sizes computed: 1,501,338 genomes
2026-07-10 12:28:27 | Running AAI query...
100% ▕████████████████████████████████████████████████████████████▏ 
2026-07-10 12:39:49 | AAI written to /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/aai/aai.parquet


## Cluster sequences by AAI using `mcl`

Treat genomes in the same cluster as "similar", they should not be present in both train and test datasets.

### Filter AAI table using a cutoff to make an edge table for MCL
The filter will be (at least 40% AAI) **and** (at least 16 shared proteins **or** (at least 20% of the proteins in the query and target genomes are shared)). Genomes that do not meet these criteria will not be connected by an edge in the table and will be considered "distinct".

In [8]:
import os
os.environ["POLARS_MAX_THREADS"] = str(50)
import polars as pl
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [9]:
MIN_GENES = 16
MIN_SHARED = 0.2
MIN_AAI = 0.4

In [10]:
AAI_TABLE = AAI_OUTDIR.joinpath("aai.parquet")

In [24]:
filtered_df = (
    pl.scan_parquet(AAI_TABLE)
    .with_columns(
        # All sequences from genomad contain "fragment" in their headers, proGenomes does not
        pl.when(pl.col("query_genome").str.contains('fragment'))
        .then(pl.lit("genomad"))
        .otherwise(pl.lit("progenomes"))
        .alias('query_genome_dataset'),
        pl.when(pl.col("target_genome").str.contains('fragment'))
        .then(pl.lit("genomad"))
        .otherwise(pl.lit("progenomes"))
        .alias('target_genome_dataset')
    )
    .filter(
        (pl.col("query_genome") != pl.col("target_genome"))
    )
    .filter(
        (
            (pl.col("shared_genes") >= MIN_GENES)
            | (
                (pl.col("query_shared") >= MIN_SHARED)
                & (pl.col("target_shared") >= MIN_SHARED)
            )
        )
        & (pl.col("aai") >= MIN_AAI)
    )
    .with_columns(
        score=pl.col("aai") * pl.min_horizontal("query_shared", "target_shared")
    )
    .select("query_genome", "target_genome", "score")
)

In [25]:
MCL_ABC = AAI_OUTDIR.joinpath("aai_mcl.abc")

In [26]:
filtered_df.sink_csv(MCL_ABC, include_header=False, separator="\t")

In [27]:
! head {MCL_ABC}
! tail {MCL_ABC}
! wc -l {MCL_ABC}

258533.SAMEA3138996.CCBB010000001~1-3205355~chromosome_mixed	266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	0.020206308
522373.SAMEA1705934.AM743169~1-4851126~chromosome_mixed	266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	0.0968734
1441629.SAMN02641561.CP007039~1-5986012~chromosome_mixed	266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	0.09677201
751586.SAMN02469913.GL883086~1-2631526~chromosome_mixed	266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	0.045351483
1121898.SAMN02441179.AUGP01000017~788-692471~standalone_virus	266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	0.00398995
1703962.SAMN03998639.CP013505~1-4473645~chromosome_mixed	266265.SAMN02598451.CP000270~1-4895836~chromosome_mixed	0.063429095
1729689.SAMN04102799.LMVC01000001~1-1011761~chromosome_mixed	266265.SAMN02598451.CP000270~1-4895836~chromosome_mixed	0.0059867552
1736509.SAMN04155778.LMJJ01000003~1-384699~chromosome_mixed	266265.SAMN02598451.CP000270~1-4895836~chromosome_mixed	0

### Run `mcl`

Using an inflation parameter `-I` of `2.0`.

In [28]:
! mcxload -abc {MCL_ABC} -o {AAI_OUTDIR.joinpath("aai_mcl.mci")} -write-tab {AAI_OUTDIR.joinpath("aai_mcl.mcxtab")}

.................................................. 1M
.................................................. 2M
.................................................. 3M
.................................................. 4M
.................................................. 5M
.................................................. 6M
.................................................. 7M
.................................................. 8M
.................................................. 9M
.................................................. 10M
.................................................. 11M
.................................................. 12M
.................................................. 13M
.................................................. 14M
.................................................. 15M
.................................................. 16M
.................................................. 17M
.................................................. 18M
...................

    nohup mcl \
        /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/aai/aai_mcl.mci \
        -I 2.0 \
        -te 50 \
        -o /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/aai/aai_mcl.clusters \
        -use-tab /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/processing/genomad_progenomes_combined/aai/aai_mcl.mcxtab \
        > ./logs/train_test_split/aai_mcl.log &

In [11]:
MCL_CLUSTERS = AAI_OUTDIR.joinpath("aai_mcl.clusters")

In [30]:
! head {MCL_CLUSTERS}
! tail {MCL_CLUSTERS}
! wc -l {MCL_CLUSTERS}

266264.SAMN02598450.CP000352~1-3928089~chromosome_mixed	522373.SAMEA1705934.AM743169~1-4851126~chromosome_mixed	1441629.SAMN02641561.CP007039~1-5986012~chromosome_mixed	751586.SAMN02469913.GL883086~1-2631526~chromosome_mixed	1703962.SAMN03998639.CP013505~1-4473645~chromosome_mixed	266265.SAMN02598451.CP000270~1-4895836~chromosome_mixed	266265.SAMN02598451.CP000271~1-3363523~chromosome_mixed	883126.SAMN02463908.JH992923~1-1219890~chromosome_mixed	266265.SAMN02598451.CP000272~1-1471779~chromosome_mixed	335284.SAMN02598327.CP000323~1-3059876~chromosome_mixed	381666.SAMEA3283071.AM260479~1-4052032~chromosome_mixed	1395516.SAMN02471806.CM002330~1-4617471~chromosome_only	267608.SAMEA3138192.AL646052~1-3716413~chromosome_mixed	718251.SAMN02603331.CP002154~1-3684607~chromosome_mixed	936455.SAMN02441117.KI421499~1-9823581~chromosome_mixed	267608.SAMEA3138192.AL646053~1-2094509~chromosome_mixed	377629.SAMN02603454.CP001614~1-5193164~chromosome_mixed	269796.SAMN02598538.CP000230~1-4352825~chromos

## Load genome clusters and split data into train and test

Ensuring:
1. Train and test genomes/contigs are distinct using the results above
2. All proteins encoded on the same contig are moved together as one unit (i.e. no single contig has proteins in both train and test)
3. Training dataset is balanced
4. Multiple distributions of source and pos/neg for test data
5. Target train/test split fraction is 0.9/0.1

The script `train_test_split.py` located in `accessory_scripts` will be used for this.

In [12]:
TRAIN_DATA_PRE = GENOMAD_PROGENOMES_COMBINED.joinpath("training_data_pre_split.parquet")

In [13]:
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data")
SPLIT_DIR = ROOT_DIR.joinpath("train_test_splits")
SPLIT_DIR.mkdir(exist_ok=True)

In [14]:
TARGET_TRAIN_FRAC = 0.90

In [15]:
! python3 {SCRIPTS_DIR.joinpath("train_test_split.py")} \
    --input {TRAIN_DATA_PRE} \
    --input-format parquet \
    --cluster-path {MCL_CLUSTERS} \
    --outdir {SPLIT_DIR} \
    --seed 20260413 \
    --train-balance \
    --train-frac {TARGET_TRAIN_FRAC} \
    --n-jobs 10 \
    --allow-downsize

2026-07-11 14:08:40 | INFO | Loaded 1273407 contigs from cluster map
2026-07-11 14:08:40 | INFO | Splitting train/test (block-atomic)...
2026-07-11 14:08:59 | INFO | Initial train proteins: 15143230
2026-07-11 14:08:59 | INFO | Initial test_pool proteins: 1682581
2026-07-11 14:09:15 | INFO | Leakage-pruned test_pool proteins: 1682581
2026-07-11 14:09:15 | INFO | Building provirus test set (chromosome_mixed contigs with both Host and Virus)...
2026-07-11 14:09:15 | INFO | test_provirus: proteins=75260 contigs=3069 V=0.575 H=0.421 M=0.004
2026-07-11 14:09:15 | INFO | Wrote /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_provirus.parquet
2026-07-11 14:09:15 | INFO | Bounding base test_pool viral fraction (block-atomic) for non-extreme configs...
2026-07-11 14:09:22 | INFO | Base v-bounded test_pool proteins: 1682581
2026-07-11 14:09:22 | INFO | Balancing training set (contig-atomic)...
2026-07-11 14:09:37 | INFO | Balanced train proteins: 13999957
2026

## Check for leakage across train and tests

Based on contig IDs and presence in the same 'genome block' (AAI similar genomes).

In [16]:
import pandas as pd

In [17]:
train_test_datasets = {}

for split_file in SPLIT_DIR.glob("*.parquet"):
    dataset_name = split_file.stem
    train_test_datasets[dataset_name] = pd.read_parquet(split_file).sort_values(["Dataset", "Contig", "contig_pos_start", "contig_pos_end"])

In [18]:
from __future__ import annotations

from typing import Dict, Any, Optional, Tuple, List
import hashlib

def read_mcl_clusters_to_contig_map(
    mcl_path: str,
    sep: str = "\t",
    keep_examples_per_cluster: int = 5,
) -> Tuple[Dict[str, int], Dict[int, int], Dict[int, List[str]]]:
    contig_to_cluster: Dict[str, int] = {}
    cluster_sizes: Dict[int, int] = {}
    cluster_examples: Dict[int, List[str]] = {}

    with open(mcl_path, "r") as fh:
        for cid, raw in enumerate(fh):
            line = raw.rstrip("\n")
            if not line:
                continue
            parts = [p.strip() for p in line.split(sep) if p.strip()]
            if not parts:
                continue

            cluster_sizes[cid] = len(parts)
            if keep_examples_per_cluster > 0:
                cluster_examples[cid] = parts[:keep_examples_per_cluster]

            for ctg in parts:
                if ctg in contig_to_cluster:
                    prev = contig_to_cluster[ctg]
                    raise AssertionError(
                        f"Contig appears in multiple MCL clusters: contig={ctg!r}, prev_cluster={prev}, new_cluster={cid}"
                    )
                contig_to_cluster[ctg] = cid

    return contig_to_cluster, cluster_sizes, cluster_examples

def sanity_check_splits(
    train_df: pd.DataFrame,
    test_dfs: Dict[str, pd.DataFrame],
    aai_mcl_cluster_path: Optional[str] = None,
    source_col: str = "Source",
    contig_col: str = "Contig",
    protein_col: str = "Protein",
    allow_test_test_overlap: bool = True,
    enforce_no_contig_reuse_across_sources: bool = True,
    check_atomicity_within_each_test: bool = True,
    max_examples: int = 10,
) -> Dict[str, Any]:
    """
    Required (train vs each test):
      - No identical proteins in train and test.
      - No identical contigs in train and test.
      - If enforce_no_contig_reuse_across_sources=True: no (Source,Contig) unit overlap train vs test.

    Atomicity:
      - Contigs are atomic between train and the UNION of all tests (a contig may appear in multiple test sets).
      - Proteins are atomic between train and the UNION of all tests.

      - With check_atomicity_within_each_test=True: within each test set, a contig's proteins
        don't get split across multiple split labels inside that test (should be true).

    MCL clusters:
      - If aai_mcl_cluster_path provided: no MCL cluster has contigs split across train and ANY test (union).

    Notes:
      - allow_test_test_overlap=True permits reusing contigs/proteins across multiple test datasets.
    """
    if train_df is None or len(train_df) == 0:
        raise ValueError("train_df is empty.")
    for col in (contig_col, protein_col):
        if col not in train_df.columns:
            raise ValueError(f"train_df missing required column: {col!r}")
    if enforce_no_contig_reuse_across_sources and source_col not in train_df.columns:
        raise ValueError(f"train_df missing required column: {source_col!r}")

    def _as_str_series(df: pd.DataFrame, col: str) -> pd.Series:
        return df[col].dropna().astype(str)

    def _unique_set(df: pd.DataFrame, col: str) -> set:
        return set(_as_str_series(df, col).unique().tolist())

    def _unit_ids(df: pd.DataFrame) -> set:
        tmp = df[[source_col, contig_col]].dropna()
        src = tmp[source_col].astype(str)
        ctg = tmp[contig_col].astype(str)
        return set((src + "\t" + ctg).tolist())

    def _hash64(s: str) -> int:
        b = s.encode("utf-8")
        return int.from_bytes(hashlib.sha256(b).digest()[:8], "little", signed=False)

    def _preview(items: set) -> List[str]:
        return list(items)[:max_examples]

    summary: Dict[str, Any] = {"train": {}, "tests": {}, "union": {}}

    train_prots = _unique_set(train_df, protein_col)
    train_contigs = _unique_set(train_df, contig_col)
    train_units = _unit_ids(train_df) if enforce_no_contig_reuse_across_sources else set()

    summary["train"] = {
        "n_ptns": int(len(train_df)),
        "n_proteins_unique": int(len(train_prots)),
        "n_contigs_unique": int(len(train_contigs)),
        "n_units_unique": int(len(train_units)) if enforce_no_contig_reuse_across_sources else None,
    }

    # Train vs each test
    seen_test_prots: set = set()
    seen_test_contigs: set = set()
    union_test_prots: set = set()
    union_test_contigs: set = set()
    union_test_units: set = set()

    for name, df in test_dfs.items():
        if df is None or len(df) == 0:
            summary["tests"][name] = {
                "n_ptns": 0,
                "n_proteins_unique": 0,
                "n_contigs_unique": 0,
                "train_overlap_proteins": 0,
                "train_overlap_contigs": 0,
                "train_overlap_units": 0 if enforce_no_contig_reuse_across_sources else None,
                "test_test_overlap_proteins": 0 if not allow_test_test_overlap else None,
                "test_test_overlap_contigs": 0 if not allow_test_test_overlap else None,
            }
            continue

        for col in (contig_col, protein_col):
            if col not in df.columns:
                raise ValueError(f"{name} missing required column: {col!r}")
        if enforce_no_contig_reuse_across_sources and source_col not in df.columns:
            raise ValueError(f"{name} missing required column: {source_col!r}")

        test_prots = _unique_set(df, protein_col)
        test_contigs = _unique_set(df, contig_col)
        test_units = _unit_ids(df) if enforce_no_contig_reuse_across_sources else set()

        prot_overlap = train_prots & test_prots
        contig_overlap = train_contigs & test_contigs
        unit_overlap = train_units & test_units if enforce_no_contig_reuse_across_sources else set()

        if not allow_test_test_overlap:
            prot_overlap_tt = seen_test_prots & test_prots
            contig_overlap_tt = seen_test_contigs & test_contigs
        else:
            prot_overlap_tt = set()
            contig_overlap_tt = set()

        summary["tests"][name] = {
            "n_ptns": int(len(df)),
            "n_proteins_unique": int(len(test_prots)),
            "n_contigs_unique": int(len(test_contigs)),
            "train_overlap_proteins": int(len(prot_overlap)),
            "train_overlap_contigs": int(len(contig_overlap)),
            "train_overlap_units": int(len(unit_overlap)) if enforce_no_contig_reuse_across_sources else None,
            "test_test_overlap_proteins": int(len(prot_overlap_tt)) if not allow_test_test_overlap else None,
            "test_test_overlap_contigs": int(len(contig_overlap_tt)) if not allow_test_test_overlap else None,
        }

        assert not prot_overlap, (
            f"{name}: protein leakage train<->test: {len(prot_overlap)} overlapping proteins. "
            f"Examples: {_preview(prot_overlap)}"
        )
        assert not contig_overlap, (
            f"{name}: contig leakage train<->test: {len(contig_overlap)} overlapping contigs. "
            f"Examples: {_preview(contig_overlap)}"
        )
        if enforce_no_contig_reuse_across_sources:
            assert not unit_overlap, (
                f"{name}: unit leakage train<->test: {len(unit_overlap)} overlapping (Source\\tContig) units. "
                f"Examples: {_preview(unit_overlap)}"
            )

        if not allow_test_test_overlap:
            assert not prot_overlap_tt, (
                f"{name}: test-test protein overlap found (disallowed): {len(prot_overlap_tt)}. "
                f"Examples: {_preview(prot_overlap_tt)}"
            )
            assert not contig_overlap_tt, (
                f"{name}: test-test contig overlap found (disallowed): {len(contig_overlap_tt)}. "
                f"Examples: {_preview(contig_overlap_tt)}"
            )

        seen_test_prots |= test_prots
        seen_test_contigs |= test_contigs

        union_test_prots |= test_prots
        union_test_contigs |= test_contigs
        union_test_units |= test_units

        # Optional: per-test internal contig atomicity
        if check_atomicity_within_each_test:
            tmp = df[[contig_col, protein_col]].dropna()
            tmp[contig_col] = tmp[contig_col].astype(str)
            tmp[protein_col] = tmp[protein_col].astype(str)
            # within a single test dataset, a contig should map to only one dataset label (itself)

    summary["union"] = {
        "n_proteins_unique_tests_union": int(len(union_test_prots)),
        "n_contigs_unique_tests_union": int(len(union_test_contigs)),
        "n_units_unique_tests_union": int(len(union_test_units)) if enforce_no_contig_reuse_across_sources else None,
    }

    # Atomicity only between train and UNION(test)
    # If a contig appears in both train and tests_union, the earlier assertions would have already fired.
    # Still, verify that no contig/protein is assigned to both labels at once.
    all_rows = []
    all_rows.append(train_df[[contig_col, protein_col]].copy().assign(_split="train"))
    if len(union_test_contigs) > 0:
        # build union df by concatenating test frames (but drop duplicates to keep size controlled)
        union_frames = []
        for name, df in test_dfs.items():
            if df is None or len(df) == 0:
                continue
            union_frames.append(df[[contig_col, protein_col]].copy())
        union_df = pd.concat(union_frames, axis=0, ignore_index=True)
        union_df = union_df.dropna(subset=[contig_col, protein_col])
        union_df[contig_col] = union_df[contig_col].astype(str)
        union_df[protein_col] = union_df[protein_col].astype(str)
        union_df = union_df.drop_duplicates(subset=[contig_col, protein_col])
        all_rows.append(union_df.assign(_split="test_union"))

    all_splits = pd.concat(all_rows, axis=0, ignore_index=True)
    all_splits = all_splits.dropna(subset=[contig_col, protein_col])
    all_splits[contig_col] = all_splits[contig_col].astype(str)
    all_splits[protein_col] = all_splits[protein_col].astype(str)

    contig_split_counts = all_splits.groupby(contig_col, sort=False)["_split"].nunique()
    bad_contigs = contig_split_counts[contig_split_counts > 1]
    if len(bad_contigs) > 0:
        bad_list = bad_contigs.index.tolist()[:max_examples]
        detail = (
            all_splits[all_splits[contig_col].isin(bad_list)]
            .groupby(contig_col)["_split"]
            .apply(lambda x: sorted(set(x.tolist())))
            .to_dict()
        )
        raise AssertionError(
            f"Contig atomicity violated between train and test_union: {len(bad_contigs)} contigs in both. "
            f"Examples (contig -> splits): {detail}"
        )

    prot_split_counts = all_splits.groupby(protein_col, sort=False)["_split"].nunique()
    bad_prots = prot_split_counts[prot_split_counts > 1]
    if len(bad_prots) > 0:
        bad_list = bad_prots.index.tolist()[:max_examples]
        detail = (
            all_splits[all_splits[protein_col].isin(bad_list)]
            .groupby(protein_col)["_split"]
            .apply(lambda x: sorted(set(x.tolist())))
            .to_dict()
        )
        raise AssertionError(
            f"Protein atomicity violated between train and test_union: {len(bad_prots)} proteins in both. "
            f"Examples (protein -> splits): {detail}"
        )

    # MCL cluster integrity: train vs UNION(test)
    if aai_mcl_cluster_path is not None:
        contig_to_cluster, cluster_sizes, cluster_examples = read_mcl_clusters_to_contig_map(aai_mcl_cluster_path)

        train_ctg_set = set(train_contigs)
        test_ctg_set = set(union_test_contigs)

        train_cluster_ids = {contig_to_cluster[c] for c in train_ctg_set if c in contig_to_cluster}
        test_cluster_ids = {contig_to_cluster[c] for c in test_ctg_set if c in contig_to_cluster}
        overlap_clusters = train_cluster_ids & test_cluster_ids

        n_train_missing = int(sum(1 for c in train_ctg_set if c not in contig_to_cluster))
        n_test_missing = int(sum(1 for c in test_ctg_set if c not in contig_to_cluster))

        summary["mcl"] = {
            "n_contigs_train_missing_cluster_map": n_train_missing,
            "n_contigs_test_union_missing_cluster_map": n_test_missing,
            "n_clusters_train": int(len(train_cluster_ids)),
            "n_clusters_test_union": int(len(test_cluster_ids)),
            "n_clusters_split_train_vs_test_union": int(len(overlap_clusters)),
        }

        if len(overlap_clusters) > 0:
            bad = list(overlap_clusters)
            bad_sorted = sorted(bad, key=lambda cid: (-int(cluster_sizes.get(cid, 0)), _hash64(str(cid))))
            bad_sorted = bad_sorted[:max_examples]

            detail = {}
            for cid in bad_sorted:
                ex = cluster_examples.get(cid, [])
                keep_n = min(5, max_examples)
                train_ex = [c for c in ex if c in train_ctg_set][:keep_n]
                test_ex = [c for c in ex if c in test_ctg_set][:keep_n]
                detail[cid] = {
                    "cluster_size": int(cluster_sizes.get(cid, -1)),
                    "example_members": ex,
                    "example_in_train": train_ex,
                    "example_in_test_union": test_ex,
                }

            raise AssertionError(
                f"MCL cluster leakage train<->test_union: {len(overlap_clusters)} clusters contain contigs in both. "
                f"Examples: {detail}"
            )

    print("Sanity check passed: no train/test leakage; contigs+proteins atomic between train and test_union; no MCL cluster split train vs tests_union.")
    return summary


In [19]:
report = sanity_check_splits(
    train_df = train_test_datasets["train"],
    test_dfs = {k: v for k, v in train_test_datasets.items() if k != "train"},
    aai_mcl_cluster_path = MCL_CLUSTERS,
    allow_test_test_overlap = True,
    enforce_no_contig_reuse_across_sources = True,
    max_examples = 10
    )

Sanity check passed: no train/test leakage; contigs+proteins atomic between train and test_union; no MCL cluster split train vs tests_union.


## Report compositional summaries of training and test datasets

In [20]:
for name, df in train_test_datasets.items():
    total_proteins = df["Protein"].nunique()
    total_contigs = df["Contig"].nunique()

    print(f"Dataset: {name}")
    print(f"\tProteins (total): {total_proteins:>10,}")
    print(f"\tContigs (total):  {total_contigs:>10,}")

    prot_nunique = df.groupby("Source")["Protein"].nunique()
    contig_nunique = df.groupby("Source")["Contig"].nunique()

    print("\tProteins by source:")
    for k, v in prot_nunique.items():
        pct = (v / total_proteins) * 100 if total_proteins else 0
        print(f"\t  {k:<6} {v:>10,} ({pct:>6.2f}%)")

    print("\tContigs by source:")
    for k, v in contig_nunique.items():
        pct = (v / total_contigs) * 100 if total_contigs else 0
        print(f"\t  {k:<6} {v:>10,} ({pct:>6.2f}%)")

    print()

Dataset: test_provirus
	Proteins (total):     75,260
	Contigs (total):       3,069
	Proteins by source:
	  Host       31,670 ( 42.08%)
	  MGE           314 (  0.42%)
	  Virus      43,276 ( 57.50%)
	Contigs by source:
	  Host        3,069 (100.00%)
	  MGE             6 (  0.20%)
	  Virus       3,069 (100.00%)

Dataset: test_near_all_virus
	Proteins (total):    846,398
	Contigs (total):      83,846
	Proteins by source:
	  Host       65,650 (  7.76%)
	  MGE        30,098 (  3.56%)
	  Virus     750,650 ( 88.69%)
	Contigs by source:
	  Host       10,305 ( 12.29%)
	  MGE         2,123 (  2.53%)
	  Virus      73,034 ( 87.10%)

Dataset: test_host_enriched
	Proteins (total):    313,192
	Contigs (total):      36,853
	Proteins by source:
	  Host      220,972 ( 70.55%)
	  MGE        39,149 ( 12.50%)
	  Virus      53,071 ( 16.95%)
	Contigs by source:
	  Host       26,530 ( 71.99%)
	  MGE         3,103 (  8.42%)
	  Virus       7,987 ( 21.67%)

Dataset: test_near_all_host
	Proteins (total):    667,53

### Write a tidy composition table for plotting

One row per dataset x unit (proteins/scaffolds) x source, with counts, dataset totals, and percentages.


In [ ]:
TABLES_DIR = Path("./tables/lgbm")
TABLES_DIR.mkdir(parents=True, exist_ok=True)

COMPOSITION_TABLE = TABLES_DIR.joinpath("train_test_dataset_composition.tsv")

dataset_name_map = {
    "train": "Training",
    "test_virus_enriched": "Virus enriched",
    "test_near_all_virus": "Near all virus",
    "test_half_virus_host": "Half viral/host",
    "test_equal_pos": "Equal viral/nonviral",
    "test_input": "Training distribution",
    "test_equal_source": "Equal viral/MGE/host",
    "test_host_enriched": "Host enriched",
    "test_provirus": "Integrated proviruses",
    "test_mge_enriched": "MGE enriched",
    "test_near_all_mge": "Near all MGE",
    "test_near_all_host": "Near all host",
}

source_order = ["Virus", "MGE", "Host"]

rows = []
for name, df in train_test_datasets.items():
    counts = {
        "Proteins": df.groupby("Source")["Protein"].nunique(),
        "Scaffolds": df.groupby("Source")["Contig"].nunique(),
    }
    totals = {
        "Proteins": df["Protein"].nunique(),
        "Scaffolds": df["Contig"].nunique(),
    }
    for unit in ("Proteins", "Scaffolds"):
        total = int(totals[unit])
        for source in source_order:
            n = int(counts[unit].get(source, 0))
            rows.append(
                {
                    "dataset_key": name,
                    "dataset": dataset_name_map.get(name, name),
                    "split": "train" if name == "train" else "test",
                    "unit": unit,
                    "source": source,
                    "count": n,
                    "total": total,
                    "percent": (n / total * 100) if total else 0.0,
                }
            )

composition = pd.DataFrame(rows)

dataset_order = [v for v in dataset_name_map.values() if v in set(composition["dataset"])]
composition["dataset"] = pd.Categorical(composition["dataset"], categories=dataset_order, ordered=True)
composition["unit"] = pd.Categorical(composition["unit"], categories=["Proteins", "Scaffolds"], ordered=True)
composition["source"] = pd.Categorical(composition["source"], categories=source_order, ordered=True)

composition = composition.sort_values(["dataset", "unit", "source"]).reset_index(drop=True)
composition.to_csv(COMPOSITION_TABLE, sep="\t", index=False)

print(f"Wrote {len(composition):,} rows to {COMPOSITION_TABLE}")
composition.head(12)


## Write the train and test dataset protein sequences

In [21]:
import os
from concurrent.futures import ThreadPoolExecutor
from pyfastatools import Parser

input_recs = {}
input_headers = set()
for record in Parser(COMBINED_FAA):
    input_headers.add(record.header.name)
    input_recs[record.header.name] = record

def write_faa_file(name_df_tuple):
    name, df = name_df_tuple
    out_path = os.path.join(SPLIT_DIR, f"{name}.faa")
    output_headers = set(df["Protein"].tolist())
    output_rec_names = input_headers.intersection(output_headers)
    with open(out_path, 'w') as f:
        for rec_name in output_rec_names:
            record = input_recs[rec_name]
            record.remove_stops()
            f.write(f">{record.header.name}\n{record.seq}\n")

with ThreadPoolExecutor() as executor:
    list(executor.map(write_faa_file, train_test_datasets.items()))

In [22]:
! ls -lah {SPLIT_DIR}/*.faa
! grep -c ">" {SPLIT_DIR}/*.faa
! grep ">" {SPLIT_DIR}/*.faa | head -n 3

-rw-rw-r-- 1 kosmopoulos kosmopoulos 109M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_equal_pos.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  57M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_equal_source.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos 178M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_half_virus_host.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  95M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_host_enriched.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos 278M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_input.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos  59M Jul 11 14:31 /storage2/scratch/kosmopoulos/projects/checkAMG/training_data/train_test_splits/test_mge_enriched.faa
-rw-rw-r-- 1 kosmopoulos kosmopoulos 205M Jul 11 14:31 /storage2